# Advanced Features

This notebook demonstrates pyetm's programmatic scenario management and data export functionalities, focusing on bulk operations, and data manipulation using Python code directly, rather than Excel interface. 'Advanced' features are also described in this notebook.

Check the environment setup.

In [ ]:
# Check the environment is properly configured.
from example_helpers import setup_notebook
setup_notebook()

# Load and create scenarios

#### Load
1. `load()` - Load a single scenario, used with `Scenario`
2. `load_many()` - Load many scenarios, used with `Scenarios`

#### Create from parameters
1. `create()` - Create a single scenario, used with `Scenario`
2. `create_many()` - Create many scenarios, used with `Scenarios`

#### Copy
1. `copy()` - Creates a deep copy (breaks preset link)
2. `copy_with_preset()` - Creates a copy that maintains preset link

Both methods accept optional metadata overrides (title, description, private, etc.)

In [ ]:
from pyetm.models.scenarios import Scenarios
from pyetm.models.scenario import Scenario
from pyetm.models.session import Session


loaded = Scenarios.load_many([5522, 5325])

# Copy a scenario - creates a new SavedScenario in MyETM
copied = loaded[1].copy(title="Copy example")

create_params = [
    {"area_code": "nl2019", "end_year": 2050, "title": "Scenario demo 2050"},
    {"area_code": "nl2019", "end_year": 2040, "title": "Scenario demo 2040"},
]

created = Scenarios.create_many(create_params)

session = Session.load(2658450)

scenarios = Scenarios()
scenarios.extend(loaded)
scenarios.extend(created)
scenarios.add(copied)  # Use add() for single items
scenarios.add(session)

### Interpolation

Another way to create scenarios is by 'interpolating' from other scenarios. You can read more about how interpolation works [here.](https://docs.energytransitionmodel.com/api/interpolation)

In [ ]:
scenario = scenarios[0]

# This will interpolate between a single scenario start year and end year to create 2
# new scenarios with end years 2030 and 2045.
interpolated = Scenario.interpolate(
    scenario,
    2030, 2045
)

# Interpolate from multiple saved scenarios to multiple end years and save
# with custom titles and metadata. For example:

saved_2040 = scenarios[3]
saved_2050 = scenarios[2]

interpolated = Scenario.interpolate(
    [saved_2040, saved_2050],
    2030, 2045,
    titles=["PyETM Demo 2030 (Interpolated)", "PyETM Demo 2045 (Interpolated)"],
    private=False
)

### User management

You can manage the users of a scenario using the following actions. These actions also work for Sessions, but we recommend managing access at the Scenario level.

In [ ]:
users = scenario.list_users()
for user in users:
    identifier = user.get("user_email") or user.get("user_id")
    print(f"{identifier}: {user.get('role')}")
print("---------------")

# Add users with different roles
scenario.update_users("viewer@saved.com", "viewer")
scenario.update_users("collaborator@saved.com", "collaborator")
scenario.update_users("owner@saved.com", "owner")

# List again
users = scenario.list_users()
for user in users:
    identifier = user.get("user_email") or user.get("user_id")
    print(f"{identifier}: {user.get('role')}")
print("---------------")

# Remove user from scenario
scenario.update_users("owner@saved.com", "remove")
# Update the role of a user - collaborator is now owner
scenario.update_users("collaborator@saved.com", "owner")


# List again
users = scenario.list_users()
for user in users:
    identifier = user.get("user_email") or user.get("user_id")
    print(f"{identifier}: {user.get('role')}")
print("---------------")


# The Packer

In the packer, all models are converted to dataframes for easy export to other data wrangling formats or to Excel.

Add scenarios to the packer.

**Note:** The packer accepts both `Scenario` (SavedScenario) and `Session` objects. When you add Scenario objects, they automatically delegate to their underlying Session for data operations, while preserving SavedScenario metadata for exports.

In [ ]:
from pyetm.models.scenario_packer import ScenarioPacker

packer = ScenarioPacker()
packer.add(*scenarios)

Show scenario inputs.

In [ ]:
packer.inputs(columns=["user", "default", "min", "max", "permitted_values"]).head(15)

Show custom curves.

In [ ]:
packer.custom_curves().head(20)

Show sortables.

In [ ]:
packer.sortables().head(20)

Show exports.

In [ ]:
# packer.exports().head(15) # Exports are the output curves, these take some time to generate. Uncomment to run

Export scenarios to Excel.

In [ ]:
# The flag include_input_details includes min, max and default for inputs in a separate sheet named "INPUT_DETAILS"
packer.to_excel(include_inputs=True, include_input_details=True, include_sortables=True, path="../examples/outputs/scenarios.xlsx")